# 3-2절 연습 문제 풀이

이 노트북은 3-2절 연습 문제(3-3 ~ 3-8)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch03/03-02_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y_xor = torch.tensor([[0.], [1.], [1.], [0.]])
LR = 0.01
EPOCHS = 1000

def train_quiet(model, X, Y_true, epochs=EPOCHS, learning_rate=LR, criterion=None):
    """본문 학습 루프와 같되 로그 대신 마지막 손실만 반환한다."""
    criterion = criterion or nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X), Y_true)
        loss.backward()
        optimizer.step()
    return loss.item()

def classify_ok(model, X, Y_true):
    model.eval()
    with torch.no_grad():
        return torch.equal((model(X) >= 0.5).float(), Y_true)

## 연습 문제 3-3

> [코드 3-3]의 모델 요약 정보 출력에서 `Param #`은 해당 계층의 파라미터의 수를 나타낸다.
> 시그모이드 활성화 계층을 포함한 네 개의 계층 객체마다 파라미터의 수가 어떻게 결정되었는지 설명해 보자.

### 풀이 해설

[코드 3-3]의 출력은 이렇다.

```
├─Linear: 1-1      [4, 2]      6
├─Sigmoid: 1-2     [4, 2]      --
├─Linear: 1-3      [4, 1]      3
├─Sigmoid: 1-4     [4, 1]      --
```

- **첫 번째 Linear(은닉층, 2 → 2): 6개** = 가중치 2 × 2 + 편향 2. 뉴런 두 개가 각각 입력 두 개를 받으므로
  가중치가 네 개, 뉴런마다 편향이 하나씩이라 편향이 두 개다.
- **두 번째 Linear(출력층, 2 → 1): 3개** = 가중치 2 × 1 + 편향 1.
- **두 Sigmoid 계층: `--`(없음)**. 시그모이드는 입력을 정해진 식으로 변환할 뿐 학습으로 값이 바뀌는 것이 없다.
  본문 2장의 용어로는 **함수형 계층**이고, 그래서 `Param #`이 `--`로 표시된다.

정리하면 파라미터 수는 `(입력 수 × 뉴런 수) + 뉴런 수`이고, 파라미터를 갖는 계층은 선형 계층뿐이다.
모델 전체 파라미터는 6 + 3 = 9개다.

In [2]:
model = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid())

print(f'{"계층":>10} {"파라미터 수":>10}   내역')
print('-' * 46)
for name, layer in zip(['Linear', 'Sigmoid', 'Linear', 'Sigmoid'], model):
    n = sum(p.numel() for p in layer.parameters())
    if n:
        w, b = layer.weight.numel(), layer.bias.numel()
        print(f'{name:>10} {n:10d}   가중치 {w} + 편향 {b}')
    else:
        print(f'{name:>10} {"--":>10}   학습 파라미터가 없는 함수형 계층')
print(f'\n모델 전체: {sum(p.numel() for p in model.parameters())}개')

        계층     파라미터 수   내역
----------------------------------------------
    Linear          6   가중치 4 + 편향 2
   Sigmoid         --   학습 파라미터가 없는 함수형 계층
    Linear          3   가중치 2 + 편향 1
   Sigmoid         --   학습 파라미터가 없는 함수형 계층

모델 전체: 9개


### 문제 검토

- **적절성: 적합.** 모델 요약 정보를 '읽는 법'을 익히게 하는 문제다. 파라미터 계층과 함수형 계층의 구분(2장)이
  `Param #` 열에 그대로 드러나므로, 두 장의 내용이 한 화면에서 연결된다.
- **[검토] 1장 연습 문제 3-1과 겹치는 부분이 있다.** 3-1도 계층별 파라미터 수를 세는 문제였다.
  다만 3-1은 손으로 세고 이 문제는 도구의 출력을 해석하는 것이라 방향이 반대이므로, 짝으로 두는 것이 오히려 좋다.
  3-1의 지문에 "[연습 문제 3-3]에서 같은 계산을 파이토치가 대신해 주는 것을 확인한다" 같은 연결을 넣어도 좋겠다.

## 연습 문제 3-4

> 선형 계층 객체를 만들 때 `bias=False` 인자를 지정하면 편향 없이 가중치만 파라미터로 사용하는 선형 계층이 된다.
> [코드 3-1]에서 정의한 모델 클래스를 수정해 은닉층과 출력층의 편향 파라미터를 제거한 후 모델의 학습 결과를 확인해 보자.

In [3]:
def build_mlp(bias=True):
    return nn.Sequential(
        nn.Linear(2, 2, bias=bias), nn.Sigmoid(),
        nn.Linear(2, 1, bias=bias), nn.Sigmoid(),
    )

TRIALS = 5
print(f'{"편향":>6} {"시드":>4} {"손실":>9} {"분류 성공":>9}   출력')
print('-' * 62)
for bias in (True, False):
    success = 0
    for seed in range(1, TRIALS + 1):
        torch.manual_seed(seed)
        model = build_mlp(bias)
        loss = train_quiet(model, X, Y_xor)
        ok = classify_ok(model, X, Y_xor)
        success += ok
        with torch.no_grad():
            out = [round(v, 2) for v in model(X).flatten().tolist()]
        print(f'{str(bias):>6} {seed:4d} {loss:9.4f} {str(ok):>9}   {out}')
    print(f'  -> 편향 {"사용" if bias else "제거"}: {TRIALS}회 중 성공 {success}회\n')

    편향   시드        손실     분류 성공   출력
--------------------------------------------------------------


  True    1    0.0037      True   [0.07, 0.94, 0.94, 0.06]


  True    2    0.0035      True   [0.06, 0.94, 0.94, 0.06]


  True    3    0.0038      True   [0.07, 0.93, 0.95, 0.06]


  True    4    0.0039      True   [0.07, 0.95, 0.94, 0.06]


...

 False    1    0.1815     False   [0.44, 0.96, 0.38, 0.39]


 False    2    0.1301     False   [0.1, 0.5, 0.92, 0.5]


 False    3    0.1298     False   [0.1, 0.5, 0.92, 0.5]


 False    4    0.1311     False   [0.11, 0.91, 0.5, 0.5]


 False    5    0.1777     False   [0.59, 0.5, 0.67, 0.04]
  -> 편향 제거: 5회 중 성공 0회



### 풀이 해설

**편향을 제거하면 다섯 번 모두 실패한다.** 편향이 있을 때는 다섯 번 중 네 번 성공한다.

이유는 결정 경계의 위치에 있다. 2장에서 본 대로 편향은 결정 경계를 원점에서 밀어내는 역할을 한다.
편향이 없으면 가중합이 `x1·w1 + x2·w2`가 되어, 결정 경계 `x1·w1 + x2·w2 = 0`은 **반드시 원점을 지나는 직선**이 된다.

그런데 XOR의 네 점 (0,0), (0,1), (1,0), (1,1) 중 (0,0)은 원점 그 자체다.
원점을 지나는 직선으로는 (0,0)을 어느 한쪽으로 확실히 밀어 넣을 수 없고, 은닉층의 두 뉴런이 모두 그런 처지가 된다.
게다가 입력이 (0, 0)이면 편향 없는 뉴런의 가중합은 **파라미터와 무관하게 항상 0**이라 시그모이드를 거쳐 늘 0.5를 낸다.
학습으로 바꿀 수 없는 값이 생기는 것이다.

편향이 '있으면 좋은 것'이 아니라 **없으면 표현할 수 없는 경계가 생긴다**는 것을 보여 주는 문제다.

In [4]:
# 편향이 없으면 입력 (0, 0)에 대한 은닉층 출력이 파라미터와 무관하게 항상 0.5임을 확인
for seed in (1, 2, 3):
    torch.manual_seed(seed)
    hidden = nn.Linear(2, 2, bias=False)
    with torch.no_grad():
        out = torch.sigmoid(hidden(torch.tensor([[0., 0.]])))
    print(f'시드 {seed}: 가중치 {[round(v, 2) for v in hidden.weight.flatten().tolist()]} '
          f'-> (0, 0)의 은닉층 출력 {[round(v, 4) for v in out.flatten().tolist()]}')

시드 1: 가중치 [0.36, -0.31, -0.14, 0.33] -> (0, 0)의 은닉층 출력 [0.5, 0.5]
시드 2: 가중치 [0.16, -0.17, 0.19, -0.04] -> (0, 0)의 은닉층 출력 [0.5, 0.5]
시드 3: 가중치 [-0.7, -0.56, -0.3, -0.67] -> (0, 0)의 은닉층 출력 [0.5, 0.5]


### 문제 검토

- **적절성: 적합. 3장에서 가장 값진 문제 중 하나다.** 결과가 명확하게 갈리고(편향 있음 4/5, 편향 없음 0/5),
  그 이유가 2장에서 배운 '편향은 결정 경계의 위치를 정한다'와 정확히 맞물린다.
  특히 XOR 데이터에 원점 (0, 0)이 포함되어 있어 편향의 필요성이 극적으로 드러난다. 데이터 선택이 좋다.
- **[검토] 판정 기준과 반복 횟수가 없다.** "학습 결과를 확인해 보자"만으로는 손실을 볼지 분류 결과를 볼지 알 수 없고,
  한 번만 돌려 보면 편향이 있는 경우도 실패할 수 있어(5회 중 1회 실패) 잘못된 결론에 이를 수 있다.
  2장 연습 문제 2-4가 '10번 이상 반복'을 명시해 이 문제를 피한 것과 대비된다.

**윤문안**

> **3-4**. 선형 계층 객체를 만들 때 `bias=False` 인자를 지정하면 편향 없이 가중치만 파라미터로 사용하는
> 선형 계층이 된다. [코드 3-1]에서 정의한 모델 클래스를 수정해 은닉층과 출력층의 편향 파라미터를 제거한 후,
> 파라미터 초깃값을 바꿔 가며 여러 번 학습해 네 샘플의 분류 결과를 편향이 있을 때와 비교해 보자.
>
> 힌트: 편향이 없으면 결정 경계가 반드시 원점을 지난다. XOR 데이터의 네 점 중 원점에 있는 점을 생각해 보자.

## 연습 문제 3-5

> [코드 3-1]에서 정의한 모델 클래스에서 다음과 같이 은닉층의 수와 뉴런의 수를 바꿔 가며 모델의 학습 결과를 확인해 보자.
> - 뉴런의 수가 세 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델
> - 뉴런의 수가 세 개인 은닉층 여섯 개를 포함한 다층 퍼셉트론 모델
> - 뉴런의 수가 열 개인 은닉층 하나를 가진 다층 퍼셉트론 모델
> - 뉴런의 수가 각각 세 개, 두 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델

In [5]:
CONFIGS = {
    '3, 3 (은닉층 2개)': [3, 3],
    '3 × 6 (은닉층 6개)': [3] * 6,
    '10 (은닉층 1개)': [10],
    '3, 2 (은닉층 2개)': [3, 2],
}

def build_deep_mlp(hidden_sizes):
    layers, in_features = [], 2
    for out_features in hidden_sizes:
        layers += [nn.Linear(in_features, out_features), nn.Sigmoid()]
        in_features = out_features
    layers += [nn.Linear(in_features, 1), nn.Sigmoid()]
    return nn.Sequential(*layers)

TRIALS = 5
print(f'{"구조":>20} {"파라미터":>8} {"성공":>6} {"평균 손실":>10}')
print('-' * 50)
for name, hidden_sizes in CONFIGS.items():
    success, losses = 0, []
    for seed in range(1, TRIALS + 1):
        torch.manual_seed(seed)
        model = build_deep_mlp(hidden_sizes)
        losses.append(train_quiet(model, X, Y_xor))
        success += classify_ok(model, X, Y_xor)
    n_params = sum(p.numel() for p in build_deep_mlp(hidden_sizes).parameters())
    print(f'{name:>20} {n_params:8d} {success:4d}/{TRIALS} {sum(losses) / TRIALS:10.4f}')

                  구조     파라미터     성공      평균 손실
--------------------------------------------------


       3, 3 (은닉층 2개)       25    4/5     0.0258


      3 × 6 (은닉층 6개)       73    4/5     0.0336


         10 (은닉층 1개)       41    5/5     0.0004


       3, 2 (은닉층 2개)       20    4/5     0.0349


### 풀이 해설

네 구조 모두 XOR을 풀 수 있다. 다만 안정성에서 차이가 난다.

- **뉴런 열 개짜리 은닉층 하나**가 다섯 번 모두 성공해 가장 안정적이다. 뉴런이 많을수록 여러 결정 경계를
  동시에 시도할 수 있어 초깃값이 나빠도 그중 하나가 제 역할을 한다.
- **은닉층 여섯 개**는 파라미터가 훨씬 많은데도 더 나아지지 않는다. 시그모이드 활성화 계층이 여섯 겹이라
  본문 p16이 설명한 **기울기 소실**이 나타나기 때문이다. 깊게 쌓는 것만으로는 좋아지지 않는다는 것을 보여 준다.
- 은닉층 두 개짜리 두 구조(3-3, 3-2)는 비슷하다. 본문 [코드 3-1]의 뉴런 두 개짜리 은닉층 하나보다는 안정적이다.

정리하면 이 문제의 답은 '무엇이 제일 좋은가'가 아니라, **깊이와 너비 중 어느 쪽을 늘리는지에 따라 결과가 다르고,
시그모이드를 쓰는 한 깊이를 늘리는 쪽은 한계가 있다**는 것이다. 이 관찰이 3-3절의 ReLU로 이어진다.

### 문제 검토

- **적절성: 적합.** 네 구조를 깊이와 너비가 대비되도록 고른 점이 좋다. 특히 '3 × 6'을 넣어 깊게 쌓아도
  좋아지지 않는 경우를 보게 한 것이, 바로 다음 절의 ReLU 도입 동기와 이어진다.
- **[검토] 3-4와 같은 문제를 안고 있다.** 판정 기준과 반복 횟수가 지문에 없다. XOR 학습은 초깃값에 따라
  성패가 갈리므로 한 번씩만 돌려 보면 우연한 결과로 순서가 뒤바뀐다. 실제로 각 구조를 다섯 번씩 돌려야
  위와 같은 경향이 드러난다.
- **[검토] 무엇을 비교해야 하는지도 없다.** 손실만 볼지, 분류 성공 여부를 볼지, 파라미터 수와 함께 볼지에 따라
  얻는 결론이 달라진다.

**윤문안**

> **3-5**. [코드 3-1]에서 정의한 모델 클래스에서 다음과 같이 은닉층의 수와 뉴런의 수를 바꿔 가며
> 파라미터 초깃값을 달리해 여러 번 학습하고, 네 샘플을 모두 바르게 분류하는 비율을 서로 비교해 보자.
> 각 모델의 파라미터 수도 함께 세어 보면 좋다.
> (이하 네 가지 구조는 그대로)

## 연습 문제 3-6

> XNOR 게이트는 XOR 게이트가 0을 출력할 때 1을, 1을 출력할 때 0을 출력한다.
> 2개의 출력으로 XOR 게이트와 XNOR 게이트의 결과를 동시에 예측하는 다층 퍼셉트론 모델을 만들어 보자.

In [6]:
# 첫 번째 열은 XOR, 두 번째 열은 XNOR의 정답
Y_both = torch.tensor([[0., 1.], [1., 0.], [1., 0.], [0., 1.]])

torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(2, 4), nn.Sigmoid(),
    nn.Linear(4, 2), nn.Sigmoid(),
)
loss = train_quiet(model, X, Y_both, epochs=2000)
model.eval()
with torch.no_grad():
    Y_pred = model(X)
print(f'손실 {loss:.4f}, 분류 성공 {torch.equal((Y_pred >= 0.5).float(), Y_both)}')
print(f'\n{"입력":>10} {"XOR 출력":>10} {"XNOR 출력":>10}   정답')
for x, y, t in zip(X.tolist(), Y_pred.tolist(), Y_both.tolist()):
    print(f'{str(tuple(int(v) for v in x)):>10} {y[0]:10.3f} {y[1]:10.3f}   {tuple(int(v) for v in t)}')
print(f'\n두 출력의 합: {[round(v, 3) for v in (Y_pred[:, 0] + Y_pred[:, 1]).tolist()]}')

손실 0.0006, 분류 성공 True

        입력     XOR 출력    XNOR 출력   정답
    (0, 0)      0.008      0.994   (0, 1)
    (0, 1)      0.985      0.012   (1, 0)
    (1, 0)      0.967      0.029   (1, 0)
    (1, 1)      0.036      0.969   (0, 1)

두 출력의 합: [1.002, 0.997, 0.996, 1.005]


### 풀이 해설

출력층 뉴런을 두 개로 늘리고 정답 텐서를 `(4, 2)` 형태로 만들면 된다. 2장 연습 문제 2-9와 같은 방식이다.
은닉층 뉴런은 XOR 하나만 풀 때보다 넉넉하게 네 개로 잡았다.

흥미로운 점은 **두 출력의 합이 거의 1**이 된다는 것이다. XNOR은 XOR의 반대이므로 두 뉴런이 서로 반대로 움직이는
관계를 스스로 학습한 결과다. 즉 이 모델은 사실상 하나의 판단을 두 방향으로 내보내고 있다.

이는 다음 절에서 배울 소프트맥스의 성질과 닿아 있다. 소프트맥스는 출력의 합이 정확히 1이 되도록 강제하는데,
여기서는 모델이 학습으로 비슷한 상태에 도달한 셈이다.

### 문제 검토

- **적절성: 적합.** 2장 연습 문제 2-9(AND와 OR 동시 출력)의 다층 퍼셉트론 판이라 난도가 자연스럽게 이어진다.
  XOR과 XNOR이 서로 반대 관계라 두 출력의 합이 1에 가까워지는 현상까지 관찰할 수 있어, 3-3절의 소프트맥스와
  원-핫 인코딩으로 넘어가는 다리가 된다.
- **[검토] 은닉층 구조를 정해 주지 않았다.** 독자가 [코드 3-1]의 뉴런 두 개짜리 은닉층을 그대로 쓰면
  출력이 두 개로 늘어난 만큼 학습이 더 어려워져 실패하기 쉽다. "은닉층의 뉴런 수는 넉넉하게 잡아 보자" 정도의
  안내가 있으면 좋다.
- **[검토] 관찰 지점을 짚어 주면 좋다.** 두 출력의 합이 1에 가까워진다는 점은 이 문제에서 가장 재미있는 관찰인데
  지문이 가리키지 않는다.

**윤문안**

> **3-6**. XNOR 게이트는 XOR 게이트가 0을 출력할 때 1을, 1을 출력할 때 0을 출력한다.
> 2개의 출력으로 XOR 게이트와 XNOR 게이트의 결과를 동시에 예측하는 다층 퍼셉트론 모델을 만들어 보자.
> 학습을 마친 뒤 두 출력값을 더해 보고, 그 값이 왜 그렇게 나오는지 생각해 보자.

## 연습 문제 3-7

> 깃허브 저장소의 data 디렉터리에는 [그림 3-2]와 비슷한 분포의 데이터가 저장되어 있는 ch3_exercise_1.csv 파일과
> ch3_exercise_2.csv 파일이 있다. 두 파일의 데이터를 각각 분류할 수 있는 두 개의 모델을 만들어 보자.
>
> 힌트: 이 파일의 정답은 숫자가 아니라 '바깥 원', '안쪽 원'과 같은 문자열이다.
> 본문에서 설명한 것처럼 클래스마다 인덱스를 부여해 변환한 후 사용해야 한다.

In [7]:
import csv

DATA_DIR = '../../data'

def load_csv(file_name):
    """CSV 파일을 읽어 입력 텐서와 정답 텐서, 클래스 이름 목록을 돌려준다."""
    with open(f'{DATA_DIR}/{file_name}', 'r') as f:
        rows = list(csv.DictReader(f))
    class_names = sorted({row['label'] for row in rows})       # 문자열 정답을 정렬해 인덱스 부여
    class_index = {name: i for i, name in enumerate(class_names)}
    X = torch.tensor([[float(row['x']), float(row['y'])] for row in rows])
    Y = torch.tensor([class_index[row['label']] for row in rows])
    return X, Y, class_names

def split_data(X, Y, train_ratio=0.6, seed=SEED):
    """무작위로 섞어 훈련과 평가 데이터로 나눈다."""
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(X), generator=generator)
    X, Y = X[order], Y[order]
    train_size = int(len(X) * train_ratio)
    return X[:train_size], Y[:train_size], X[train_size:], Y[train_size:]

In [8]:
HIDDEN_DIM = 16

def build_classifier(num_classes, hidden_dim=HIDDEN_DIM):
    # 교차 엔트로피 손실 함수를 쓰므로 출력층에 활성화 계층을 두지 않는다
    return nn.Sequential(
        nn.Linear(2, hidden_dim), nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        nn.Linear(hidden_dim, num_classes),
    )

def run_classification(file_name, hidden_dim=HIDDEN_DIM, epochs=1000):
    X_all, Y_all, class_names = load_csv(file_name)
    X_train, Y_train, X_test, Y_test = split_data(X_all, Y_all)
    torch.manual_seed(SEED)
    model = build_classifier(len(class_names), hidden_dim)
    loss = train_quiet(model, X_train, Y_train, epochs=epochs,
                       criterion=nn.CrossEntropyLoss())
    model.eval()
    with torch.no_grad():
        accuracy = (model(X_test).argmax(dim=-1) == Y_test).float().mean().item() * 100
    print(f'{file_name}: 샘플 {len(X_all)}개, 클래스 {class_names}')
    print(f'    훈련 손실 {loss:.4f}, 평가 정확도 {accuracy:.2f}%')
    return accuracy

for file_name in ('ch3_exercise_1.csv', 'ch3_exercise_2.csv'):
    run_classification(file_name)

ch3_exercise_1.csv: 샘플 400개, 클래스 ['바깥 원', '안쪽 원']
    훈련 손실 0.0000, 평가 정확도 100.00%


ch3_exercise_2.csv: 샘플 300개, 클래스 ['아래', '위']
    훈련 손실 0.0000, 평가 정확도 100.00%


### 풀이 해설

두 데이터 모두 **평가 정확도 100%**로 분류된다. 동심원과 반달 모양은 본문의 회오리 데이터보다 단순해서
은닉층 뉴런 16개만으로도 충분하다.

이 문제의 관문은 모델이 아니라 **데이터 준비**다. 세 단계가 필요하다.

1. 정답이 '바깥 원' 같은 문자열이므로 클래스마다 인덱스를 매긴다. 여기서는 정렬한 순서대로 0, 1을 부여했다.
2. 열 이름이 본문 예제(`x1`, `x2`)와 달리 `x`, `y`다. 본문 [코드 3-7]을 그대로 복사하면 `KeyError`가 난다.
3. 파일에 클래스가 순서대로 모여 있으므로 그대로 앞뒤로 자르면 훈련 데이터에 한 클래스만 들어간다.
   반드시 섞은 뒤에 나눠야 한다.

교차 엔트로피 손실 함수를 사용했으므로 출력층에 활성화 계층을 두지 않았고, 정답도 원-핫이 아닌 인덱스 그대로 썼다.

### 문제 검토

- **적절성: 적합.** 본문 예제와 같은 구조의 모델을 새 데이터에 적용해 보게 하는 실전형 문제다.
  문자열 정답을 인덱스로 바꾸는 과정은 실제 데이터를 다룰 때 반드시 거치는 단계라 연습 가치가 높다.
- **[중요] 열 이름이 본문 예제와 다르다.** 본문 [코드 3-7]은 `row['x1']`, `row['x2']`로 읽는데
  이 파일들의 열 이름은 `x`, `y`다. 지문이 "x 좌표의 값(`x`), y 좌표의 값(`y`), 정답(`label`)"이라고
  밝히고 있으므로 읽는 독자는 알 수 있지만, 코드를 복사해 쓰면 `KeyError`를 만난다.
  본문 예제 파일(`ch3_spiral_data.csv`)은 `x1`, `x2`를 쓰므로 **연습 문제 데이터의 열 이름을 `x1`, `x2`로 맞추면**
  독자가 걸려 넘어질 일이 없다. 데이터 파일을 고칠 수 없다면 지문에서 한 번 더 강조하는 편이 좋다.
- **[검토] 데이터를 섞어야 한다는 안내가 없다.** 두 파일 모두 같은 클래스끼리 모여 있어서, 본문 예제처럼
  앞에서부터 6:4로 자르면 훈련 데이터에 한 클래스만 들어간다. 힌트에 한 줄 덧붙이면 좋다.

**윤문안 (힌트)**

> 힌트: 이 파일의 정답은 숫자가 아니라 '바깥 원', '안쪽 원'과 같은 문자열이다. 본문에서 설명한 것처럼 클래스마다
> 인덱스를 부여해 변환한 후 사용해야 한다. 또한 파일에는 같은 클래스의 샘플이 모여 있으므로,
> 훈련 데이터와 평가 데이터로 나누기 전에 순서를 무작위로 섞어야 한다.

## 연습 문제 3-8 [도전 문제]

> 본문의 은닉층 역할 설명을 바탕으로, 두 개의 은닉층을 포함한 다층 퍼셉트론에서 첫 번째와 두 번째 은닉층
> 뉴런의 수가 각각 세 개, 두 개인 모델과 두 개, 세 개인 모델의 차이를 유추해 보자.
>
> 힌트: 각 은닉층이 만드는 결정 경계의 수와 은닉 공간의 차원 수에 주목해 보자.

### 풀이 해설

본문 p14의 설명을 두 모델에 그대로 적용해 보면 된다. 은닉층의 뉴런 수는 **그 계층이 만드는 결정 경계의 수**이자,
**다음 계층이 선을 긋게 될 은닉 공간의 차원 수**다.

**3 → 2 모델**
- 첫 번째 은닉층: 입력 데이터 공간(2차원)에 결정 경계 **세 개**를 긋는다.
- 두 번째 은닉층: 그 결과가 만드는 **3차원** 은닉 공간에 결정 경계 두 개를 긋는다.
- 즉 넓게 펼쳐 놓고(3차원) 그중에서 골라내는(2개) 구조다.

**2 → 3 모델**
- 첫 번째 은닉층: 데이터 공간에 결정 경계 **두 개**만 긋는다.
- 두 번째 은닉층: **2차원** 은닉 공간에 결정 경계 세 개를 긋는다.
- 즉 처음에 2차원으로 좁혀 놓고(2개) 그 안에서 여러 번 자르는(3개) 구조다.

핵심은 **첫 번째 은닉층이 만든 차원 수가 그 뒤 모든 계층의 상한이 된다**는 점이다.
2 → 3 모델에서 두 번째 은닉층이 아무리 여러 번 선을 그어도, 이미 2차원으로 눌려 버린 정보는 되살아나지 않는다.
첫 번째 은닉층에서 서로 다른 두 입력이 같은 좌표로 겹쳐졌다면, 뒤에서는 영영 구분할 수 없다.
반면 3 → 2 모델은 일단 3차원으로 펼친 뒤 줄이므로 정보를 잃을 위험이 적다.

그래서 일반적으로 다층 퍼셉트론은 **앞쪽 은닉층을 넓게 두고 뒤로 갈수록 좁히는** 구조를 많이 쓴다.
아래 코드로 두 구조를 여러 번 학습해 비교해 볼 수 있다.

In [9]:
SHAPES = {'3 -> 2': [3, 2], '2 -> 3': [2, 3]}
TRIALS = 10

print(f'{"구조":>8} {"파라미터":>8} {"성공":>7} {"평균 손실":>10}')
print('-' * 38)
for name, hidden_sizes in SHAPES.items():
    success, losses = 0, []
    for seed in range(1, TRIALS + 1):
        torch.manual_seed(seed)
        model = build_deep_mlp(hidden_sizes)
        losses.append(train_quiet(model, X, Y_xor))
        success += classify_ok(model, X, Y_xor)
    n_params = sum(p.numel() for p in build_deep_mlp(hidden_sizes).parameters())
    print(f'{name:>8} {n_params:8d} {success:5d}/{TRIALS} {sum(losses) / TRIALS:10.4f}')

      구조     파라미터      성공      평균 손실
--------------------------------------


  3 -> 2       20     7/10     0.0469


  2 -> 3       19     5/10     0.0718


### 문제 검토

- **적절성: 적합. 도전 문제로 잘 설계됐다.** 코드를 돌리지 않고 본문의 은닉 공간 설명만으로 추론하게 하는데,
  힌트가 '결정 경계의 수'와 '은닉 공간의 차원 수'라는 두 축을 정확히 짚어 줘서 길을 잃지 않는다.
  3-2절에서 가장 설명이 깊었던 은닉 공간 개념을 활용하는 마무리 문제로 알맞다.
- **[검토] 확인할 방법을 덧붙이면 좋다.** 유추한 내용을 실험으로 확인해 볼 수 있다는 안내가 있으면,
  추론에서 그치지 않고 검증까지 이어진다. 다만 XOR 데이터로는 두 구조의 차이가 뚜렷하게 드러나지 않으므로
  (위 실험에서도 성공 횟수가 비슷하다), '차이가 크지 않다면 그 이유도 생각해 보자'까지 덧붙이면 더 좋다.
  이 데이터는 네 점뿐이라 2차원으로 눌러도 잃을 정보가 거의 없기 때문이다.